# Day 10 — CuTe DSL Hello World

**Goal:** launch your first CuTe-DSL kernel from Python and have every
thread write its global linear ID into a tensor.

Two decorators do the work:

| Decorator      | Runs on | Purpose                                              |
|----------------|---------|------------------------------------------------------|
| `@cute.jit`    | Host    | A regular Python function that may *launch* kernels. |
| `@cute.kernel` | Device  | The GPU kernel body — one instance per thread.       |

Inside a `@cute.kernel`:

```python
tidx, tidy, tidz = cute.arch.thread_idx()    # like CUDA threadIdx
bidx, bidy, bidz = cute.arch.block_idx()     # like CUDA blockIdx
bdim, _,    _    = cute.arch.block_dim()     # like CUDA blockDim
cute.printf("fmt {} {}", a, b)               # device-side printf
gA[i] = value                                # tensor store
```

DSL 4.4+ traces a plain Python `if cond:` as a dynamic GPU branch
automatically — no `cutlass.dynamic_expr(...)` wrapper needed.


In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch

cutlass.cuda.initialize_cuda_context()
print("device:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))


## Part 1 — `hello_kernel`

**Task:** only thread 0 of block 0 calls `cute.printf` — otherwise we'd
see 32 lines, one per thread in the warp.


In [ ]:
@cute.kernel
def hello_kernel():
    # TODO: only have thread 0 of block 0 print.
    raise NotImplementedError


@cute.jit
def hello_world():
    hello_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))


# Uncomment once you've filled in the kernel:
# hello_world()


## Part 2 — `write_tid_kernel`

**Task:** each thread computes its global linear id and stores it into
`gA[gid]`. Cast `gid` to `gA.element_type` before the store. Guard with
`if gid < cute.size(gA):` so a too-large grid doesn't overrun the tensor.


In [ ]:
@cute.kernel
def write_tid_kernel(gA: cute.Tensor):
    # TODO: each thread writes its global linear id to gA[gid].
    raise NotImplementedError


@cute.jit
def write_thread_id(mA: cute.Tensor):
    threads_per_block: cutlass.Constexpr = 128
    n = cute.size(mA)
    num_blocks = (n + threads_per_block - 1) // threads_per_block
    write_tid_kernel(mA).launch(
        grid=(num_blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


In [ ]:
# Verify against torch.arange once you've filled in write_tid_kernel.
N = 1024
a = torch.full((N,), -1, device="cuda", dtype=torch.int32)
# write_thread_id(from_dlpack(a))
# torch.cuda.synchronize()
# torch.testing.assert_close(a, torch.arange(N, device="cuda", dtype=torch.int32))
# print("first 8:", a[:8].tolist())


---

## Reference solution

Spoilers below. Re-run these cells to overwrite the TODO definitions.


In [ ]:
@cute.kernel
def hello_kernel():
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    if tidx == 0 and bidx == 0:
        cute.printf("hello from CuTe DSL: tidx=%d bidx=%d", tidx, bidx)


@cute.kernel
def write_tid_kernel(gA: cute.Tensor):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    gid = bidx * bdim + tidx
    if gid < cute.size(gA):
        gA[gid] = gid.to(gA.element_type)


hello_world()

a = torch.full((N,), -1, device="cuda", dtype=torch.int32)
write_thread_id(from_dlpack(a))
torch.cuda.synchronize()
torch.testing.assert_close(a, torch.arange(N, device="cuda", dtype=torch.int32))
print("first 8:", a[:8].tolist())


## What you should walk away knowing

- The host/device split: `@cute.jit` traces Python and emits IR;
  `@cute.kernel` becomes a CUDA function with one invocation per thread.
- `cute.printf` is device-side and asynchronous — it may print *after*
  the Python interpreter has continued.
- `from_dlpack(torch_tensor)` is the bridge from PyTorch into CuTe.
- The unit test for a CuTe kernel is just `torch.testing.assert_close`
  against a reference torch expression.
